# Clone your voice with VoxCPM2 (Kaggle)

Use this notebook again whenever you want a new recording: add your clean voice reference to a Kaggle Dataset, change the text in the **TEXT TO SPEAK** cell, then run the cells below it. The generated WAV is saved under `/kaggle/working/voice_clones/` and can be downloaded from Kaggle's Files panel.

> Only use a voice recording you own or have permission to use. A clean, single-speaker recording gives the best match.

## One-time Kaggle setup
1. In Notebook **Settings**, choose a GPU accelerator if one is available.
2. Add your voice recording as a Kaggle Dataset using **Add Input**. Prefer a clear, dry recording with no music or other speakers.
3. Run the cells from top to bottom. For future voiceovers, edit only the text cell and run the generation and save cells again.
4. If the internet is disabled in notebook settings, turn it on for the first run so the model and packages can download.


In [ ]:
%pip install -q voxcpm soundfile "huggingface_hub>=1.16,<2"


## Load VoxCPM2
This downloads/loads the model once per Kaggle session. `load_denoiser=False` keeps memory use lower; the voice sample should already be clean.

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from voxcpm import VoxCPM

# Load the pretrained voice-cloning model. On a GPU this is usually faster.
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SAMPLE_RATE = model.tts_model.sample_rate
print("VoxCPM2 is ready. Sample rate:", SAMPLE_RATE)


## Choose your reference voice recording
Set `REFERENCE_AUDIO` to the WAV file inside your Kaggle input dataset. The example below matches the dataset path used in your earlier notebook. If you attach a different dataset, inspect `/kaggle/input` and update this path.

In [ ]:
# Path to the clean recording of YOUR voice in the Kaggle dataset.
REFERENCE_AUDIO = Path("/kaggle/input/datasets/jarvissilva/sample-voice/sample.wav")

# Show available files if the path above needs changing.
if not REFERENCE_AUDIO.exists():
    print("Reference path not found. Audio files currently available under /kaggle/input:")
    for path in Path("/kaggle/input").rglob("*"):
        if path.suffix.lower() in {".wav", ".mp3", ".flac", ".m4a", ".ogg"}:
            print(path)
    raise FileNotFoundError("Update REFERENCE_AUDIO above to one of the listed audio files.")

print("Reference recording:", REFERENCE_AUDIO)
display(Audio(filename=str(REFERENCE_AUDIO)))


## TEXT TO SPEAK — edit this each time
Replace the example below with the words you want the cloned voice to say. Keep punctuation in the text; it helps the model produce natural pauses. The notebook does not need a transcript of the reference recording for this VoxCPM2 workflow.

In [ ]:
TEXT = "Hello! This is a voice-cloned sample. Replace this sentence with your own text."

# Leave the rest of this cell unchanged.
if not TEXT.strip():
    raise ValueError("Please enter the text you want the voice to say in TEXT.")
print("Text to generate:\n", TEXT.strip())


## Generate the voice recording
These are the settings that worked for the approved sample. Generation can take a little while, especially on CPU. Avoid starting multiple generation cells at once to prevent GPU memory errors.

In [ ]:
# Ask VoxCPM2 to speak your text in the reference speaker's voice.
# Keep these values as-is for the settings used in your successful sample.
audio = model.generate(
    text=TEXT.strip(),
    reference_wav_path=str(REFERENCE_AUDIO),
    normalize=False,
    cfg_value=2.0,
    inference_timesteps=20,
)

# Convert the result to a standard mono NumPy audio array.
audio = np.asarray(audio).squeeze()
if audio.ndim != 1 or audio.size == 0:
    raise RuntimeError("The model returned empty or unexpected audio. Please rerun generation once.")
print(f"Generated {audio.size / SAMPLE_RATE:.1f} seconds of audio.")
display(Audio(audio, rate=SAMPLE_RATE))


## Save and download the WAV
The filename includes the current time so each run creates a separate file instead of replacing an earlier voice clone. In Kaggle, open the right-side **Files** panel, expand `working/voice_clones`, then download the WAV.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/voice_clones")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = OUTPUT_DIR / f"my_voice_{timestamp}.wav"
sf.write(str(output_path), audio, SAMPLE_RATE, subtype="PCM_16")
print("Saved WAV file:", output_path)
print(f"Duration: {audio.size / SAMPLE_RATE:.1f} seconds | Sample rate: {SAMPLE_RATE} Hz")


## If you run into a problem
- **Reference file not found:** Add the dataset with **Add Input**, then copy the printed audio path into `REFERENCE_AUDIO`.
- **CUDA out of memory:** Make sure no other generation is running; use a shorter text per generation or switch to CPU (slower).
- **Voice sounds unlike you:** Use a longer, clean sample with only your voice, minimal echo/noise, and natural speech. The reference clip is a guide, not a guarantee of an exact match.
- **Need another take:** Edit `TEXT`, then rerun the generation and save cells. The model can stay loaded while this Kaggle session remains active.
